<a href="https://colab.research.google.com/github/thanhnhan959604/MusicEmotionProject/blob/main/musicEmotion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/drive/MyDrive/MusicEmotion/features.zip -d /content/drive/MyDrive/MusicEmotion/

unzip:  cannot find or open /content/drive/MyDrive/MusicEmotion/features.zip, /content/drive/MyDrive/MusicEmotion/features.zip.zip or /content/drive/MyDrive/MusicEmotion/features.zip.ZIP.


In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ==========================================
# 1. CẤU HÌNH HỆ THỐNG
# ==========================================
BASE_DIR = '/content/drive/MyDrive/MusicEmotion'
CSV_PATH = f'{BASE_DIR}/DATASET_VIETNAMESE_ONLY.csv'
CLEANED_CSV_PATH = f'{BASE_DIR}/DATASET_CLEANED_TOP_85.csv'
TEXT_DIR = f'{BASE_DIR}/text_features'
AUDIO_DIR = f'{BASE_DIR}/audio_features'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Đang chạy trên thiết bị: {device}")

# ==========================================
# 2. KIẾN TRÚC MÔ HÌNH (CACHE LÊN RAM)
# ==========================================
class MultimodalEmotionDataset(Dataset):
    def __init__(self, csv_path, text_dir, audio_dir):
        df = pd.read_csv(csv_path)
        self.data = []

        print("[*] Đang kéo toàn bộ dữ liệu từ Google Drive thả vào RAM (Chỉ mất 1 lần chờ)...")
        # Nạp tất cả lên RAM 1 lần duy nhất. Cực nhanh lúc Train!
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Đọc file .npy"):
            track_id = str(row['Spotify_ID'])
            val, eng = row['valence'], row['energy']
            t_path, a_path = os.path.join(text_dir, f"{track_id}.npy"), os.path.join(audio_dir, f"{track_id}.npy")

            if os.path.exists(t_path) and os.path.exists(a_path):
                self.data.append({
                    'id': track_id,
                    'text_feat': torch.from_numpy(np.load(t_path)).float(),
                    'audio_feat': torch.from_numpy(np.load(a_path)).float(),
                    'label': torch.tensor([val, eng]).float()
                })

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

class CrossAttentionMER(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, hidden_dim=512, dropout=0.3):
        super().__init__()
        self.t2a = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.a2t = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_t2a, self.norm_a2t = nn.LayerNorm(embed_dim), nn.LayerNorm(embed_dim)

        self.fc1 = nn.Linear(embed_dim * 2, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.norm2 = nn.LayerNorm(hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, 2)
        self.relu, self.dropout = nn.ReLU(), nn.Dropout(dropout)

    def forward(self, text, audio):
        text_seq, audio_seq = text.unsqueeze(1), audio.unsqueeze(1)
        t2a_out, _ = self.t2a(query=text_seq, key=audio_seq, value=audio_seq)
        t2a_out = self.norm_t2a(t2a_out + text_seq).squeeze(1)

        a2t_out, _ = self.a2t(query=audio_seq, key=text_seq, value=text_seq)
        a2t_out = self.norm_a2t(a2t_out + audio_seq).squeeze(1)

        x = self.dropout(self.relu(self.norm1(self.fc1(torch.cat((t2a_out, a2t_out), dim=1)))))
        x = self.dropout(self.relu(self.norm2(self.fc2(x))))
        return torch.sigmoid(self.out(x))

# ==========================================
# 3. HUẤN LUYỆN NHANH "GIÁM KHẢO"
# ==========================================
print("\n[*] Bước 1: Chuẩn bị dữ liệu để đào tạo Giám khảo...")
dataset = MultimodalEmotionDataset(CSV_PATH, TEXT_DIR, AUDIO_DIR)
# Dùng bộ nhớ RAM nội bộ nên tăng num_workers không cần thiết, batch_size có thể để lớn cho lẹ
loader = DataLoader(dataset, batch_size=64, shuffle=True)

model = CrossAttentionMER(dropout=0.2).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

epochs = 30
print(f"\n[*] Bước 2: Đang huấn luyện Giám khảo ({epochs} Epochs)...")
model.train()

# Lúc này do đã có sẵn trên RAM, epoch sẽ bay cái vèo
for epoch in range(epochs):
    total_loss = 0
    loop = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

    for batch in loop:
        t, a, y = batch['text_feat'].to(device), batch['audio_feat'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        preds = model(t, a)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch+1) % 5 == 0:
        print(f"-> Hoàn tất Epoch {epoch+1}/{epochs} | Loss trung bình: {total_loss/len(loader):.4f}")

# ==========================================
# 4. CHẤM ĐIỂM VÀ LỌC OUTLIER
# ==========================================
print("\n[*] Bước 3: Đang chấm điểm và tìm Kẻ mạo danh (Noisy Labels)...")
model.eval()
errors = {}

with torch.no_grad():
    for item in tqdm(dataset, desc="Đang chấm điểm"):
        t = item['text_feat'].unsqueeze(0).to(device)
        a = item['audio_feat'].unsqueeze(0).to(device)
        y_true = item['label'].numpy()
        track_id = item['id']

        pred = model(t, a).cpu().numpy()[0]

        err_val = abs(pred[0] - y_true[0])
        err_eng = abs(pred[1] - y_true[1])
        errors[track_id] = err_val + err_eng

df_full = pd.read_csv(CSV_PATH)
df_valid = df_full[df_full['Spotify_ID'].astype(str).isin(errors.keys())].copy()
df_valid['Model_Error'] = df_valid['Spotify_ID'].astype(str).map(errors)
df_valid = df_valid.sort_values(by='Model_Error')

KEEP_RATIO = 0.85
keep_count = int(len(df_valid) * KEEP_RATIO)
df_cleaned = df_valid.head(keep_count)

df_cleaned = df_cleaned.drop(columns=['Model_Error'])
df_cleaned.to_csv(CLEANED_CSV_PATH, index=False, encoding='utf-8-sig')

# ==========================================
# 5. BÁO CÁO KẾT QUẢ
# ==========================================
print("\n" + "="*50)
print(f"✅ ĐÃ THANH LỌC THÀNH CÔNG!")
print(f"Tổng số bài gốc có npy  : {len(df_valid)}")
print(f"Số bài được giữ (85%)   : {len(df_cleaned)}")
print(f"Đã XÓA BỎ               : {len(df_valid) - len(df_cleaned)} bài hát gây nhiễu.")
print(f"File siêu sạch lưu tại  : {CLEANED_CSV_PATH}")
print("="*50)

[*] Đang chạy trên thiết bị: cuda

[*] Bước 1: Chuẩn bị dữ liệu để đào tạo Giám khảo...
[*] Đang kéo toàn bộ dữ liệu từ Google Drive thả vào RAM (Chỉ mất 1 lần chờ)...


Đọc file .npy: 100%|██████████| 2286/2286 [08:31<00:00,  4.47it/s]



[*] Bước 2: Đang huấn luyện Giám khảo (30 Epochs)...


-> Hoàn tất Epoch 5/30 | Loss trung bình: 0.0148


-> Hoàn tất Epoch 10/30 | Loss trung bình: 0.0106


...

-> Hoàn tất Epoch 20/30 | Loss trung bình: 0.0055


-> Hoàn tất Epoch 25/30 | Loss trung bình: 0.0040


-> Hoàn tất Epoch 30/30 | Loss trung bình: 0.0035

[*] Bước 3: Đang chấm điểm và tìm Kẻ mạo danh (Noisy Labels)...


Đang chấm điểm: 100%|██████████| 2286/2286 [00:03<00:00, 713.14it/s]



✅ ĐÃ THANH LỌC THÀNH CÔNG!
Tổng số bài gốc có npy  : 2286
Số bài được giữ (85%)   : 1943
Đã XÓA BỎ               : 343 bài hát gây nhiễu.
File siêu sạch lưu tại  : /content/drive/MyDrive/MusicEmotion/DATASET_CLEANED_TOP_85.csv


In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
from tqdm import tqdm

# ==========================================
# 1. CẤU HÌNH HỆ THỐNG
# ==========================================
BASE_DIR = '/content/drive/MyDrive/MusicEmotion'
CSV_PATH = f'{BASE_DIR}/DATASET_CLEANED_TOP_85.csv'
TEXT_DIR = f'{BASE_DIR}/text_features'
AUDIO_DIR = f'{BASE_DIR}/audio_features'
MODEL_SAVE_PATH = f'{BASE_DIR}/best_decoupled_mer_mixup.pth'

BATCH_SIZE = 5         # Xử lý 1 lần 5 bài theo đúng thiết lập an toàn
K_FOLDS = 5
EPOCHS_PER_FOLD = 45
FINAL_EPOCHS = 60
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-3
DROPOUT_RATE = 0.4
MIXUP_ALPHA = 0.4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Bắt đầu khởi động hệ thống trên: {device.type.upper()}\n")

# ==========================================
# 2. DATASET VÀ MÔ HÌNH TÁCH LUỒNG (DECOUPLED HEADS)
# ==========================================
class MultimodalEmotionDataset(Dataset):
    def __init__(self, csv_path, text_dir, audio_dir):
        df = pd.read_csv(csv_path)
        self.data = []
        for _, row in df.iterrows():
            track_id = str(row['Spotify_ID'])
            val, eng = row['valence'], row['energy']
            t_path, a_path = os.path.join(text_dir, f"{track_id}.npy"), os.path.join(audio_dir, f"{track_id}.npy")

            if os.path.exists(t_path) and os.path.exists(a_path):
                self.data.append({
                    'text_feat': torch.from_numpy(np.load(t_path)).float(),
                    'audio_feat': torch.from_numpy(np.load(a_path)).float(),
                    'label': torch.tensor([val, eng]).float()
                })

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]['text_feat'], self.data[idx]['audio_feat'], self.data[idx]['label']

class DecoupledCrossAttentionMER(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, hidden_dim=512, dropout=0.4):
        super(DecoupledCrossAttentionMER, self).__init__()

        # 1. Tầng Giao tiếp chung
        self.t2a = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.a2t = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_t2a = nn.LayerNorm(embed_dim)
        self.norm_a2t = nn.LayerNorm(embed_dim)

        # 2. Luồng chuyên gia Valence (Text-Heavy)
        self.valence_head = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

        # 3. Luồng chuyên gia Energy (Audio-Heavy)
        self.energy_head = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, text, audio):
        text_seq, audio_seq = text.unsqueeze(1), audio.unsqueeze(1)

        t2a_out, _ = self.t2a(query=text_seq, key=audio_seq, value=audio_seq)
        t2a_out = self.norm_t2a(t2a_out + text_seq).squeeze(1)

        a2t_out, _ = self.a2t(query=audio_seq, key=text_seq, value=text_seq)
        a2t_out = self.norm_a2t(a2t_out + audio_seq).squeeze(1)

        # Tách luồng chuyên môn
        val_input = torch.cat((text, t2a_out), dim=1)
        eng_input = torch.cat((audio, a2t_out), dim=1)

        val_pred = self.valence_head(val_input)
        eng_pred = self.energy_head(eng_input)

        out = torch.cat((val_pred, eng_pred), dim=1)
        return torch.sigmoid(out)

# ==========================================
# 3. THUẬT TOÁN MIXUP DATA AUGMENTATION
# ==========================================
def mixup_data(text, audio, y, alpha=0.4, device='cuda'):
    '''Trộn ngẫu nhiên các bài hát trong cùng 1 Batch'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = text.size()[0]
    index = torch.randperm(batch_size).to(device)

    mixed_text = lam * text + (1 - lam) * text[index, :]
    mixed_audio = lam * audio + (1 - lam) * audio[index, :]
    mixed_y = lam * y + (1 - lam) * y[index, :]

    return mixed_text, mixed_audio, mixed_y

# ==========================================
# 4. K-FOLD ĐÁNH GIÁ CHUYÊN SÂU
# ==========================================
def evaluate_kfold_mixup(dataset):
    print(f"🔄 Bắt đầu chiến dịch K-Fold với DECOUPLED HEADS + MIXUP...")
    kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    fold_results = {'val_r2': [], 'eng_r2': [], 'val_mae': [], 'eng_mae': []}

    for fold, (train_ids, test_ids) in enumerate(kfold.split(dataset)):
        print(f"\n" + "="*15 + f" ĐANG CHẠY FOLD {fold+1}/{K_FOLDS} " + "="*15)

        train_loader = DataLoader(Subset(dataset, train_ids), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
        test_loader = DataLoader(Subset(dataset, test_ids), batch_size=BATCH_SIZE, shuffle=False)

        # Gọi kiến trúc mới
        model = DecoupledCrossAttentionMER(dropout=DROPOUT_RATE).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_PER_FOLD)

        model.train()
        for epoch in range(EPOCHS_PER_FOLD):
            for text_feat, audio_feat, labels in train_loader:
                text_feat, audio_feat, labels = text_feat.to(device), audio_feat.to(device), labels.to(device)
                text_mixed, audio_mixed, labels_mixed = mixup_data(text_feat, audio_feat, labels, MIXUP_ALPHA, device)

                optimizer.zero_grad()
                outputs = model(text_mixed, audio_mixed)
                loss = criterion(outputs, labels_mixed)
                loss.backward()
                optimizer.step()
            scheduler.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for text_feat, audio_feat, labels in test_loader:
                outputs = model(text_feat.to(device), audio_feat.to(device))
                all_preds.append(outputs.cpu().numpy())
                all_labels.append(labels.numpy())

        all_preds, all_labels = np.vstack(all_preds), np.vstack(all_labels)
        r2_val, r2_eng = r2_score(all_labels[:, 0], all_preds[:, 0]), r2_score(all_labels[:, 1], all_preds[:, 1])

        print(f"✔️ Fold {fold+1} | Valence R2: {r2_val:.4f} - Energy R2: {r2_eng:.4f}")

        fold_results['val_r2'].append(r2_val)
        fold_results['eng_r2'].append(r2_eng)

    print("\n" + "★"*50)
    print("BÁO CÁO KẾT QUẢ CUỐI CÙNG (TRUNG BÌNH 5-FOLD)")
    print("★"*50)
    print(f"[VALENCE] R2 Trung bình: {np.mean(fold_results['val_r2']):.4f}")
    print(f"[ENERGY]  R2 Trung bình: {np.mean(fold_results['eng_r2']):.4f}")

# ==========================================
# 5. HUẤN LUYỆN MÔ HÌNH FINAL (100% DATA)
# ==========================================
def train_ultimate_model(dataset):
    print("\n" + "🔥" * 20)
    print("HUẤN LUYỆN MÔ HÌNH FINAL ĐỂ DEMO")
    print("🔥" * 20)

    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    model = DecoupledCrossAttentionMER(dropout=DROPOUT_RATE).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=FINAL_EPOCHS)

    model.train()
    for epoch in range(FINAL_EPOCHS):
        for text_feat, audio_feat, labels in loader:
            text_feat, audio_feat, labels = text_feat.to(device), audio_feat.to(device), labels.to(device)
            t_mix, a_mix, l_mix = mixup_data(text_feat, audio_feat, labels, MIXUP_ALPHA, device)

            optimizer.zero_grad()
            outputs = model(t_mix, a_mix)
            loss = criterion(outputs, l_mix)
            loss.backward()
            optimizer.step()
        scheduler.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{FINAL_EPOCHS}] - Đang rèn luyện tư duy độc lập cho 2 luồng nơ-ron...")

    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"\n✅ ĐÃ LƯU MÔ HÌNH SIÊU CẤP TẠI: {MODEL_SAVE_PATH}")

# ==========================================
# CHẠY QUY TRÌNH
# ==========================================
if __name__ == "__main__":
    dataset = MultimodalEmotionDataset(CSV_PATH, TEXT_DIR, AUDIO_DIR)
    # Bước 1: Đánh giá sức mạnh thực tế
    evaluate_kfold_mixup(dataset)
    # Bước 2: Lưu file mô hình chốt sổ
    train_ultimate_model(dataset)

🚀 Bắt đầu khởi động hệ thống trên: CUDA

🔄 Bắt đầu chiến dịch K-Fold với DECOUPLED HEADS + MIXUP...

=============== ĐANG CHẠY FOLD 1/5 ===============
✔️ Fold 1 | Valence R2: 0.4901 - Energy R2: 0.6759

=============== ĐANG CHẠY FOLD 2/5 ===============
✔️ Fold 2 | Valence R2: 0.5721 - Energy R2: 0.6686

=============== ĐANG CHẠY FOLD 3/5 ===============
✔️ Fold 3 | Valence R2: 0.5382 - Energy R2: 0.7242

=============== ĐANG CHẠY FOLD 4/5 ===============
✔️ Fold 4 | Valence R2: 0.5113 - Energy R2: 0.7296

=============== ĐANG CHẠY FOLD 5/5 ===============
✔️ Fold 5 | Valence R2: 0.4729 - Energy R2: 0.6788

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
BÁO CÁO KẾT QUẢ CUỐI CÙNG (TRUNG BÌNH 5-FOLD)
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
[VALENCE] R2 Trung bình: 0.5169
[ENERGY]  R2 Trung bình: 0.6954

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
HUẤN LUYỆN MÔ HÌNH FINAL ĐỂ DEMO
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
Epoch [10/60] - Đang rèn luyện tư duy độc lập cho 2 luồng nơ-ron...
Epoch [20/60] - Đang rèn luy

In [ ]:
# ==========================================
# 0. KẾT NỐI GOOGLE DRIVE
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, accuracy_score
from tqdm import tqdm # Thư viện thanh tiến trình

# ==========================================
# 1. CẤU HÌNH HỆ THỐNG
# ==========================================
BASE_DIR = '/content/drive/MyDrive/MusicEmotion'
CSV_PATH = f'{BASE_DIR}/DATASET_CLEANED_TOP_85.csv'
TEXT_DIR = f'{BASE_DIR}/text_features'
AUDIO_DIR = f'{BASE_DIR}/audio_features'

BATCH_SIZE = 5
K_FOLDS = 5
EPOCHS_PER_FOLD = 45
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-3
DROPOUT_RATE = 0.4
MIXUP_ALPHA = 0.4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Bắt đầu khởi động hệ thống trên: {device.type.upper()}\n")

# ==========================================
# 2. KIẾN TRÚC MÔ HÌNH VÀ DATASET
# ==========================================
class MultimodalEmotionDataset(Dataset):
    def __init__(self, csv_path, text_dir, audio_dir):
        df = pd.read_csv(csv_path)
        self.data = []

        # [THÊM TQDM 1] - Thanh tiến trình đọc file
        print("[*] Đang kéo toàn bộ dữ liệu từ Google Drive thả vào RAM...")
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Đọc file .npy"):
            track_id = str(row['Spotify_ID'])
            val, eng = row['valence'], row['energy']
            t_path, a_path = os.path.join(text_dir, f"{track_id}.npy"), os.path.join(audio_dir, f"{track_id}.npy")

            if os.path.exists(t_path) and os.path.exists(a_path):
                self.data.append({
                    'text_feat': torch.from_numpy(np.load(t_path)).float(),
                    'audio_feat': torch.from_numpy(np.load(a_path)).float(),
                    'label': torch.tensor([val, eng]).float()
                })
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]['text_feat'], self.data[idx]['audio_feat'], self.data[idx]['label']

class DecoupledCrossAttentionMER(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, hidden_dim=512, dropout=0.4):
        super(DecoupledCrossAttentionMER, self).__init__()
        self.t2a = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.a2t = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_t2a = nn.LayerNorm(embed_dim)
        self.norm_a2t = nn.LayerNorm(embed_dim)

        self.valence_head = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.energy_head = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, text, audio):
        text_seq, audio_seq = text.unsqueeze(1), audio.unsqueeze(1)
        t2a_out, _ = self.t2a(query=text_seq, key=audio_seq, value=audio_seq)
        t2a_out = self.norm_t2a(t2a_out + text_seq).squeeze(1)
        a2t_out, _ = self.a2t(query=audio_seq, key=text_seq, value=text_seq)
        a2t_out = self.norm_a2t(a2t_out + audio_seq).squeeze(1)

        val_input = torch.cat((text, t2a_out), dim=1)
        eng_input = torch.cat((audio, a2t_out), dim=1)

        val_pred = self.valence_head(val_input)
        eng_pred = self.energy_head(eng_input)
        return torch.sigmoid(torch.cat((val_pred, eng_pred), dim=1))

def mixup_data(text, audio, y, alpha=0.4, device='cuda'):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    batch_size = text.size()[0]
    index = torch.randperm(batch_size).to(device)
    return lam * text + (1 - lam) * text[index, :], lam * audio + (1 - lam) * audio[index, :], lam * y + (1 - lam) * y[index, :]

# ==========================================
# 3. K-FOLD KẾT HỢP ĐÁNH GIÁ PHÂN LOẠI
# ==========================================
def evaluate_all_metrics(dataset):
    print(f"\n🔄 Bắt đầu K-Fold đánh giá toàn diện (Hồi quy & Phân loại)...")
    kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

    results = {'r2_val': [], 'r2_eng': [], 'acc_val': [], 'acc_eng': [], 'acc_4_class': []}

    for fold, (train_ids, test_ids) in enumerate(kfold.split(dataset)):
        print(f"\n" + "="*15 + f" ĐANG CHẠY FOLD {fold+1}/{K_FOLDS} " + "="*15)
        train_loader = DataLoader(Subset(dataset, train_ids), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
        test_loader = DataLoader(Subset(dataset, test_ids), batch_size=BATCH_SIZE, shuffle=False)

        model = DecoupledCrossAttentionMER(dropout=DROPOUT_RATE).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_PER_FOLD)

        # Traning (Có Mixup)
        model.train()
        for epoch in range(EPOCHS_PER_FOLD):
            # [THÊM TQDM 2] - Thanh tiến trình cho từng Epoch
            loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS_PER_FOLD}", leave=False)

            for text_feat, audio_feat, labels in loop:
                t, a, l = text_feat.to(device), audio_feat.to(device), labels.to(device)
                t_mix, a_mix, l_mix = mixup_data(t, a, l, MIXUP_ALPHA, device)
                optimizer.zero_grad()
                loss = criterion(model(t_mix, a_mix), l_mix)
                loss.backward()
                optimizer.step()

                # Cập nhật số Loss liên tục lên thanh tiến trình
                loop.set_postfix(loss=loss.item())

            scheduler.step()

        # Testing (Đánh giá thực tế)
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            # [THÊM TQDM 3] - Thanh tiến trình lúc Test
            for text_feat, audio_feat, labels in tqdm(test_loader, desc="Đang đánh giá", leave=False):
                outputs = model(text_feat.to(device), audio_feat.to(device))
                all_preds.append(outputs.cpu().numpy())
                all_labels.append(labels.numpy())

        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)

        # ---------------------------------------------------------
        # A. TÍNH CHỈ SỐ HỒI QUY (Regression)
        # ---------------------------------------------------------
        r2_val = r2_score(all_labels[:, 0], all_preds[:, 0])
        r2_eng = r2_score(all_labels[:, 1], all_preds[:, 1])

        # ---------------------------------------------------------
        # B. TÍNH CHỈ SỐ PHÂN LOẠI (Classification)
        # ---------------------------------------------------------
        pred_val_class = (all_preds[:, 0] >= 0.5).astype(int)
        true_val_class = (all_labels[:, 0] >= 0.5).astype(int)

        pred_eng_class = (all_preds[:, 1] >= 0.5).astype(int)
        true_eng_class = (all_labels[:, 1] >= 0.5).astype(int)

        acc_val = accuracy_score(true_val_class, pred_val_class) * 100
        acc_eng = accuracy_score(true_eng_class, pred_eng_class) * 100

        pred_quad = pred_val_class * 2 + pred_eng_class
        true_quad = true_val_class * 2 + true_eng_class
        acc_4_class = accuracy_score(true_quad, pred_quad) * 100

        print(f"✔️ R2 Hồi quy  | Val: {r2_val:.4f}  - Eng: {r2_eng:.4f}")
        print(f"✔️ Phân loại 2 Lớp (2-Cat) | Val: {acc_val:.2f}% - Eng: {acc_eng:.2f}%")
        print(f"✔️ Phân loại 4 Lớp (4-Cat) | 4 Góc phần tư: {acc_4_class:.2f}%")

        results['r2_val'].append(r2_val); results['r2_eng'].append(r2_eng)
        results['acc_val'].append(acc_val); results['acc_eng'].append(acc_eng)
        results['acc_4_class'].append(acc_4_class)

    # ---------------------------------------------------------
    # BÁO CÁO TỔNG KẾT
    # ---------------------------------------------------------
    print("\n" + "🔥"*25)
    print(" BÁO CÁO ĐỐI CHIẾU VỚI BÀI BÁO KHOA HỌC (5-FOLD AVG)")
    print("🔥"*25)
    print("1. KẾT QUẢ HỒI QUY (Regression):")
    print(f"   👉 [Valence] R2: {np.mean(results['r2_val']):.4f}")
    print(f"   👉 [Energy]  R2: {np.mean(results['r2_eng']):.4f}")

    print("\n2. KẾT QUẢ PHÂN LOẠI 2 LỚP (Two-category):")
    print(f"   👉 [Valence] Tích cực/Tiêu cực : {np.mean(results['acc_val']):.2f} %")
    print(f"   👉 [Energy]  Sôi động/Êm dịu    : {np.mean(results['acc_eng']):.2f} %")

    print("\n3. KẾT QUẢ PHÂN LOẠI 4 LỚP (Four-category):")
    print(f"   👉 [4 Góc Phần Tư] Dự đoán trúng : {np.mean(results['acc_4_class']):.2f} %")
    print("="*50)

if __name__ == "__main__":
    dataset = MultimodalEmotionDataset(CSV_PATH, TEXT_DIR, AUDIO_DIR)
    evaluate_all_metrics(dataset)

Mounted at /content/drive
🚀 Bắt đầu khởi động hệ thống trên: CUDA

[*] Đang kéo toàn bộ dữ liệu từ Google Drive thả vào RAM...


Đọc file .npy: 100%|██████████| 1943/1943 [00:12<00:00, 149.49it/s]


🔄 Bắt đầu K-Fold đánh giá toàn diện (Hồi quy & Phân loại)...


ValueError: Cannot have number of splits n_splits=5 greater than the number of samples: n_samples=0.